In [ ]:
!pip install  -U -q git+https://github.com/huggingface/transformers.git git+https://github.com/huggingface/trl.git datasets bitsandbytes peft qwen-vl-utils  accelerate
# Tested with transformers==4.53.0.dev0, trl==0.20.0.dev0, datasets==3.6.0, bitsandbytes==0.46.0, peft==0.15.2, qwen-vl-utils==0.0.11, wandb==0.20.1, accelerate==1.8.1

In [ ]:
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

In [ ]:
import os
from datasets import Dataset, Features, Image, Value, load_dataset, DatasetDict
from PIL import Image as PILImage
import random
import pandas as pd
from collections import defaultdict

ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")["raw"]

ds

/home/ebmi/anaconda3/envs/qwenvl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 58849
})

In [ ]:
from datasets import concatenate_datasets

# Remove invalid question entries
valid_ds = ds.filter(lambda ex: ex["question"] and ex["question"] != "none")

# Identify abnormal samples (source != 'normal')
abnormal_ids = set(
    ex["img_id"] for ex in valid_ds if ex["source"].lower() != "normal"
)

# add "Does this image contain any finding?" = "yes" for abnormal cases
seen_ids = set()
added_examples = []
for ex in valid_ds:
    if ex["img_id"] in abnormal_ids and ex["img_id"] not in seen_ids:
        added_examples.append({
            "image": ex["image"],
            "source": ex["source"],
            "question": "Does this image contain any finding?",
            "answer": "yes",
            "img_id": ex["img_id"]
        })
        seen_ids.add(ex["img_id"])

# Combine cleaned data with added questions
modified_ds = Dataset.from_list(added_examples)
cleaned_ds = concatenate_datasets([valid_ds, modified_ds])


In [ ]:
# summary
print("Original size:", len(ds))
print("Cleaned size: ", len(valid_ds))
print("final modifierd data size: ", len(cleaned_ds))
print("new added QAs: ", len(added_examples))

cleaned_ds

Original size: 58849
Cleaned size:  58798
final modifierd data size:  62747
new added QAs:  3949


Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 62747
})

In [ ]:
import random
from datasets import load_dataset, DatasetDict

# all unique img_ids
all_ids = sorted(set(cleaned_ds["img_id"]))

# Shuffle & split those IDs into 80/10/10
random.seed(42)
random.shuffle(all_ids)

n = len(all_ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
# n_test will be whatever is left
train_ids = set(all_ids[:n_train])
val_ids   = set(all_ids[n_train : n_train + n_val])
test_ids  = set(all_ids[n_train + n_val :])



In [ ]:
def filter_by_id(example, id_set):
    return example["img_id"] in id_set

train_ds = cleaned_ds.filter(lambda ex: filter_by_id(ex, train_ids),
                     batched=False)
val_ds   = cleaned_ds.filter(lambda ex: filter_by_id(ex, val_ids),
                     batched=False)
test_ds  = cleaned_ds.filter(lambda ex: filter_by_id(ex, test_ids),
                     batched=False)

In [ ]:
dataset = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})

print({k: len(v) for k, v in dataset.items()})

{'train': 50105, 'validation': 6287, 'test': 6355}


In [ ]:
# keeping copy for tracking ( image,question,answer,source,img_id)  #dataset_full['test'][i]['img_id'] (or ['source'])
dataset_full = dataset

data = dataset_full.remove_columns(["source", "img_id"])
data

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 50105
    })
    validation: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6287
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6355
    })
})

In [ ]:
data['train']

Dataset({
    features: ['image', 'question', 'answer'],
    num_rows: 50105
})

In [ ]:
train_dataset = [format_data(sample) for sample in data['train']]
eval_dataset = [format_data(sample) for sample in data['validation']]
test_dataset = [format_data(sample) for sample in data['test']]

In [ ]:
train_dataset[200]

[{'role': 'system',
  'content': [{'type': 'text', 'text': 'Answer as a medical specialist'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576>},
   {'type': 'text', 'text': 'Where in the image is the abnormality?'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': 'center; center-left; center-right; lower-center; lower-left; lower-right; upper-center; upper-left; upper-right'}]}]

In [ ]:
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,Qwen2_5_VLProcessor,
)


model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = Qwen2_5_VLProcessor.from_pretrained(model_id, use_fast=True) #otherwise autoprocessor

#even Qwen2VLProcessor would work (https://github.com/huggingface/transformers/issues/36246)

In [ ]:
train_dataset[0]

[{'role': 'system',
  'content': [{'type': 'text', 'text': 'Answer as a medical specialist'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576>},
   {'type': 'text',
    'text': 'Are there any abnormalities in the image? Check all that are present.'}]},
 {'role': 'assistant',
  'content': [{'type': 'text', 'text': 'ulcerative colitis'}]}]

In [ ]:
train_dataset[0][1:2]

[{'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576>},
   {'type': 'text',
    'text': 'Are there any abnormalities in the image? Check all that are present.'}]}]

In [ ]:
# !pip install qwen_vl_utils

In [ ]:
from qwen_vl_utils import process_vision_info


def generate_text_from_sample(model, processor, sample, max_new_tokens=1024, device="cuda"):
    # Prepare the text input by applying the chat template
    text_input = processor.apply_chat_template(
        sample[1:2], tokenize=False, add_generation_prompt=True  # Use the sample without the system message
    )

    # Process the visual input from the sample
    image_inputs, _ = process_vision_info(sample)

    # Prepare the inputs for the model
    model_inputs = processor(
        text=[text_input],
        images=image_inputs,
        return_tensors="pt",
    ).to(
        device
    )  # Move inputs to the specified device

    # Generate text with the model
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Trim the generated ids to remove the input ids
    trimmed_generated_ids = [out_ids[len(in_ids) :] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)]

    # Decode the output text
    output_text = processor.batch_decode(
        trimmed_generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    return output_text[0]  # Return the first decoded output text

In [ ]:
# Example of how to call the method with sample:
output = generate_text_from_sample(model, processor, train_dataset[0])
output

'To determine if there are any abnormalities in the image, we need to carefully examine the tissue and any visible structures within it. Here are the steps to follow:\n\n1. **Check for any irregularities in the tissue texture or color.**\n2. **Look for any unusual lesions, ulcers, or growths.**\n3. **Examine the blood vessels for any abnormalities.**\n4. **Check for any changes in the overall appearance of the tissue.**\n\nUpon examining the image, I do not see any obvious abnormalities such as lesions, ulcers, or growths. The tissue appears to be relatively uniform in texture and color. The blood vessels also appear to be normal.\n\nTherefore, based on the visual inspection, there do not appear to be any abnormalities present in the image.'

In [ ]:
import gc
import time


def clear_memory():
    # Delete variables if they exist in the current global scope
    if "inputs" in globals():
        del globals()["inputs"]
    if "model" in globals():
        del globals()["model"]
    if "processor" in globals():
        del globals()["processor"]
    if "trainer" in globals():
        del globals()["trainer"]
    if "peft_model" in globals():
        del globals()["peft_model"]
    if "bnb_config" in globals():
        del globals()["bnb_config"]
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


clear_memory()

GPU allocated memory: 0.02 GB
GPU reserved memory: 2.52 GB


### PEFT

In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,Qwen2_5_VLProcessor,
    BitsAndBytesConfig,
)

# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16, quantization_config=bnb_config
)
processor = Qwen2_5_VLProcessor.from_pretrained(model_id, use_fast= True)

In [ ]:
for name, module in model.named_modules():
    print(name)


model
model.visual
model.visual.patch_embed
model.visual.patch_embed.proj
model.visual.rotary_pos_emb
model.visual.blocks
model.visual.blocks.0
model.visual.blocks.0.norm1
model.visual.blocks.0.norm2
model.visual.blocks.0.attn
model.visual.blocks.0.attn.qkv
model.visual.blocks.0.attn.proj
model.visual.blocks.0.mlp
model.visual.blocks.0.mlp.gate_proj
model.visual.blocks.0.mlp.up_proj
model.visual.blocks.0.mlp.down_proj
model.visual.blocks.0.mlp.act_fn
model.visual.blocks.1
model.visual.blocks.1.norm1
model.visual.blocks.1.norm2
model.visual.blocks.1.attn
model.visual.blocks.1.attn.qkv
model.visual.blocks.1.attn.proj
model.visual.blocks.1.mlp
model.visual.blocks.1.mlp.gate_proj
model.visual.blocks.1.mlp.up_proj
model.visual.blocks.1.mlp.down_proj
model.visual.blocks.1.mlp.act_fn
model.visual.blocks.2
model.visual.blocks.2.norm1
model.visual.blocks.2.norm2
model.visual.blocks.2.attn
model.visual.blocks.2.attn.qkv
model.visual.blocks.2.attn.proj
model.visual.blocks.2.mlp
model.visual.bloc

In [ ]:
for name, module in model.named_modules():
    if "proj" in name:
        print(name)

model.visual.patch_embed.proj
model.visual.blocks.0.attn.proj
model.visual.blocks.0.mlp.gate_proj
model.visual.blocks.0.mlp.up_proj
model.visual.blocks.0.mlp.down_proj
model.visual.blocks.1.attn.proj
model.visual.blocks.1.mlp.gate_proj
model.visual.blocks.1.mlp.up_proj
model.visual.blocks.1.mlp.down_proj
model.visual.blocks.2.attn.proj
model.visual.blocks.2.mlp.gate_proj
model.visual.blocks.2.mlp.up_proj
model.visual.blocks.2.mlp.down_proj
model.visual.blocks.3.attn.proj
model.visual.blocks.3.mlp.gate_proj
model.visual.blocks.3.mlp.up_proj
model.visual.blocks.3.mlp.down_proj
model.visual.blocks.4.attn.proj
model.visual.blocks.4.mlp.gate_proj
model.visual.blocks.4.mlp.up_proj
model.visual.blocks.4.mlp.down_proj
model.visual.blocks.5.attn.proj
model.visual.blocks.5.mlp.gate_proj
model.visual.blocks.5.mlp.up_proj
model.visual.blocks.5.mlp.down_proj
model.visual.blocks.6.attn.proj
model.visual.blocks.6.mlp.gate_proj
model.visual.blocks.6.mlp.up_proj
model.visual.blocks.6.mlp.down_proj
mode

In [ ]:
from peft import LoraConfig, get_peft_model

# Configure LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    # target_modules=["q_proj", "v_proj"],
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# Apply PEFT model adaptation
peft_model = get_peft_model(model, peft_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

trainable params: 23,794,688 || all params: 8,315,961,344 || trainable%: 0.2861


In [ ]:
from trl import SFTConfig

# Configure training arguments
training_args = SFTConfig(
    output_dir="qwenvl2.5-7b-instruct-qlora-kvasirvqa",  # Directory to save the model
    num_train_epochs=3,  # Number of training epochs
    per_device_train_batch_size=4,  # Batch size for training
    per_device_eval_batch_size=4,  # Batch size for evaluation
    gradient_accumulation_steps=8,  # Steps to accumulate gradients
    gradient_checkpointing=True,  # Enable gradient checkpointing for memory efficiency
    # Optimizer and scheduler settings
    optim="adamw_torch_fused",  # Optimizer type
    learning_rate=1e-4,  # Learning rate for training
    lr_scheduler_type="constant",  # Type of learning rate scheduler
    # Logging and evaluation
    logging_steps=50,  # Steps interval for logging
    eval_steps=50,  # Steps interval for evaluation
    eval_strategy="steps",  # Strategy for evaluation 'epoch/steps'
    save_strategy="steps",  # Strategy for saving the model 'epoch/steps'
    save_steps=100,  # Steps interval for saving
    metric_for_best_model="eval_loss",  # Metric to evaluate the best model
    greater_is_better=False,  # Whether higher metric values are better
    load_best_model_at_end=True,  # Load the best model after training
    # Mixed precision and gradient settings
    bf16=True,  # Use bfloat16 precision
    tf32=True,  # Use TensorFloat-32 precision
    max_grad_norm=0.3,  # Maximum norm for gradient clipping
    warmup_ratio=0.03,  # Ratio of total steps for warmup
    # Hub and reporting
    push_to_hub=False,  # Whether to push model to Hugging Face Hub
    report_to="none",  # Reporting tool for tracking metrics >>wandb
    # Gradient checkpointing settings
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Options for gradient checkpointing
    # Dataset configuration
    # dataset_text_field="",  # Text field in dataset
    dataset_kwargs={"skip_prepare_dataset": True},  # Additional dataset options
    # max_seq_length=1024  # Maximum sequence length for input
)

training_args.remove_unused_columns = False  # Keep unused columns in dataset

In [ ]:
# !pip install wandb

In [ ]:
# import wandb

# wandb.init(
#     project="qwen2-7b-instruct-kvasirvqa",  # change this
#     name="qwen2-7b-instruct-trl-kvasirvqa",  # change this
#     config=training_args,
# )

In [ ]:
from transformers import Qwen2_5_VLProcessor

In [ ]:
from qwen_vl_utils import process_vision_info

# Create a data collator to encode text and image pairs
def collate_fn(examples):
    # Get the texts and images, and apply the chat template
    texts = [
        processor.apply_chat_template(example, tokenize=False) for example in examples
    ]  # Prepare texts for processing
    image_inputs = [process_vision_info(example)[0] for example in examples]  # Process the images to extract inputs

    # Tokenize the texts and process the images
    batch = processor(
        text=texts, images=image_inputs, return_tensors="pt", padding=True
    )  # Encode texts and images into tensors

    # The labels are the input_ids, and we mask the padding tokens in the loss computation
    labels = batch["input_ids"].clone()  # Clone input IDs for labels
    labels[labels == processor.tokenizer.pad_token_id] = -100  # Mask padding tokens in labels

    # Ignore the image token index in the loss computation (model specific)
    if isinstance(processor, Qwen2_5_VLProcessor):  # Check if the processor is Qwen2_5_VLProcessor
        image_tokens = [151652, 151653, 151655]  # Specific image token IDs for Qwen2_5_VLProcessor
    else:
        image_tokens = [processor.tokenizer.convert_tokens_to_ids(processor.image_token)]  # Convert image token to ID

    # Mask image token IDs in the labels
    for image_token_id in image_tokens:
        labels[labels == image_token_id] = -100  # Mask image token IDs in labels

    batch["labels"] = labels  # Add labels to the batch

    return batch  # Return the prepared batch

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
    processing_class=processor.tokenizer,
)

In [ ]:
# import torch
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(training_args.output_dir)

### test

In [ ]:
clear_memory()

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = Qwen2_5_VLProcessor.from_pretrained(model_id, use_fast=True)

Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  2.17it/s]


In [ ]:
adapter_path = "/home/ebmi/Desktop/Research/Codes/kvasir-vqa/qwenvl2.5-7b-instruct-qlora-kvasirvqa"
model.load_adapter(adapter_path)

In [ ]:
# def clean_answer(text):
#     if "Assistant:" in text:
#         return text.split("Assistant:")[-1].strip()
#     return text.strip()

In [ ]:
train_dataset[0][:2]

[{'role': 'system',
  'content': [{'type': 'text', 'text': 'Answer as a medical specialist'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576>},
   {'type': 'text',
    'text': 'Are there any abnormalities in the image? Check all that are present.'}]}]

In [ ]:
import evaluate
import torch
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import Levenshtein

In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import evaluate

preds = []
refs = []

for sample in tqdm(test_dataset):
    pred = generate_text_from_sample(model, processor, sample)
    gold = [c["text"] for m in sample if m["role"] == "assistant" for c in m["content"] if c["type"] == "text"]

    preds.append(pred)
    refs.append(gold if isinstance(gold, list) else [gold])

100%|██████████| 6355/6355 [41:23<00:00,  2.56it/s]  


In [ ]:
# Prepare flat reference list for most metrics
refs_single = [r[0] for r in refs]

# Accuracy
accuracy = sum(p in r for p, r in zip(preds, refs)) / len(refs) * 100

# Load metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

bleu_res   = bleu.compute(predictions=preds, references=[[r] for r in refs_single])
rouge_res  = rouge.compute(predictions=preds, references=refs_single)
meteor_res = meteor.compute(predictions=preds, references=refs_single)

# Jaccard Similarity
j_scores = []
for r, p in zip(refs_single, preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard = sum(j_scores) / len(j_scores) * 100

# Cosine Similarity (TF-IDF)
vectorizer = TfidfVectorizer().fit(refs_single + preds)
ref_vecs  = vectorizer.transform(refs_single)
pred_vecs = vectorizer.transform(preds)
cos_sims  = cosine_similarity(ref_vecs, pred_vecs).diagonal()
cosine = cos_sims.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_score = average_levenshtein_similarity(refs, preds)



[nltk_data] Downloading package wordnet to /home/ebmi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ebmi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ebmi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
results = {
    "accuracy (%)": round(accuracy, 2),
    "bleu": bleu_res,
    "rouge": rouge_res,
    "meteor": meteor_res,
    "jaccard (%)": round(jaccard, 2),
    "cosine (%)": round(cosine, 2),
    "levenshtein (%)": round(levenshtein_score, 2)
}

for k, v in results.items():
    print(f"{k}:\n{v}\n")

accuracy (%):
86.67

bleu:
{'bleu': 0.7979302687546428, 'precisions': [0.8838316227874461, 0.8034127767592724, 0.7595501606569083, 0.751614291863969], 'brevity_penalty': 1.0, 'length_ratio': 1.027431802552151, 'translation_length': 13446, 'reference_length': 13087}

rouge:
{'rouge1': np.float64(0.9262256609968538), 'rouge2': np.float64(0.18107987529650815), 'rougeL': np.float64(0.9243524165090375), 'rougeLsum': np.float64(0.924413116486362)}

meteor:
{'meteor': np.float64(0.535749580077325)}

jaccard (%):
89.82

cosine (%):
77.77

levenshtein (%):
90.83

